# 🏦 Bank Marketing — Customer Segmentation with K-Prototypes

## Objective
This notebook segments bank customers using **K-Prototypes clustering** to identify high-propensity groups likely to subscribe to a term deposit.

Unlike K-Means (numeric only) or K-Modes (categorical only), **K-Prototypes handles mixed-type data natively** — making it ideal for the UCI Bank Marketing dataset which contains both numeric and categorical features.

## Workflow
```
1. Load raw data
2. Define numeric & categorical features
3. Scale numeric features (StandardScaler)
4. Find optimal k using Elbow Method
5. Fit final K-Prototypes model
6. Interpret: conversion rate per cluster
7. Profile each cluster (mean + mode)
8. Visualize with heatmap
9. Name segments & export results
```

> **Note:** We cluster on the FULL dataset (all y values), then cross-tabulate with `y` afterwards.
> This preserves contrast signal — a 38% conversion cluster is only meaningful against the ~11% baseline.


## Step 1 — Install & Import Libraries

`kmodes` is the only new library needed. It provides both K-Modes (categorical only)
and K-Prototypes (mixed data). Install once in your local environment:
```bash
pip install kmodes
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully!')

## Step 2 — Load Raw Data

We use `raw_data.csv` instead of `processed_data.csv` because:
- The processed file is likely already one-hot encoded
- K-Prototypes requires original categorical string values (not dummies)
- We need full control over feature separation

> The UCI Bank dataset uses **semicolon (`;`)** as delimiter.

In [ ]:
df = pd.read_csv('raw_data.csv', sep=';')

print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nTarget distribution:\n{df["y"].value_counts()}')
print(f'\nBaseline conversion rate: {df["y"].value_counts(normalize=True)["yes"]*100:.1f}%')

## Step 3 — Define Feature Types

K-Prototypes requires you to explicitly tell it which columns are categorical
via their **index positions** in the array. We organize numerics first, then
categoricals at the end — this makes index tracking straightforward.

**Numeric features:** age, duration, campaign, pdays, previous + macroeconomic indicators

**Categorical features:** demographic info (job, marital, education) + campaign contact info

> `duration` is kept here for clustering context but remember: in a real predictive model,
> it should be dropped since it's only known after the call ends.

In [ ]:
num_cols = [
    'age', 'duration', 'campaign', 'pdays', 'previous',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
    'euribor3m', 'nr.employed'
]

cat_cols = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month',
    'day_of_week', 'poutcome'
]

# Encode target: yes=1, no=0
target = df['y'].map({'yes': 1, 'no': 0})

# Build feature matrix (numerics first, then categoricals)
X = df[num_cols + cat_cols].copy()

# Get categorical column indices — K-Prototypes needs these!
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

print(f'Numeric features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Categorical indices in X: {cat_idx}')

## Step 4 — Scale Numeric Features

K-Prototypes computes dissimilarity as:
```
Total dissimilarity = Euclidean distance (numeric) + gamma × Hamming distance (categorical)
```

If numeric features are not scaled, large-magnitude columns (e.g., `nr.employed` ~5000)
will dominate the Euclidean part and effectively drown out categorical information.

**StandardScaler** transforms each numeric column to mean=0, std=1, balancing their contribution.

> Categorical columns are NOT scaled — they use Hamming distance (0 if same, 1 if different).

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])

print('Numeric features scaled to mean=0, std=1')
print(X_scaled[num_cols].describe().loc[['mean','std']].round(3))

## Step 5 — Find Optimal k with Elbow Method

K-Prototypes minimizes a **cost function** (total within-cluster dissimilarity).
We test k=2 to k=8 and plot the cost — the 'elbow' point where cost stops
dropping sharply is the optimal k.

**`init='Cao'`**: Uses the Cao initialization method, which is more deterministic
and converges faster than random init for mixed-type data.

> ⚠️ This cell can take 3–8 minutes depending on your machine since it fits 7 models.
> Reduce `k_range` if needed.

In [ ]:
costs = []
k_range = range(2, 9)

for k in k_range:
    print(f'Fitting k={k}...', end=' ')
    kproto = KPrototypes(n_clusters=k, init='Cao', random_state=42, n_jobs=-1)
    kproto.fit(X_scaled, categorical=cat_idx)
    costs.append(kproto.cost_)
    print(f'cost={kproto.cost_:.2f}')

# Plot elbow
plt.figure(figsize=(8, 4))
plt.plot(list(k_range), costs, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Cost (Total Dissimilarity)', fontsize=12)
plt.title('Elbow Method — K-Prototypes', fontsize=14)
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_kproto.png', dpi=150)
plt.show()
print('Elbow plot saved as elbow_kproto.png')

## Step 6 — Fit Final K-Prototypes Model

Set `k_optimal` based on the elbow plot above. For the UCI Bank dataset,
**k=4** typically works well — it's interpretable (4 distinct customer personas)
without over-segmenting.

The model returns a **cluster label (0 to k-1)** for every row in the dataset.

In [ ]:
k_optimal = 4  # ← adjust based on your elbow plot

print(f'Fitting final model with k={k_optimal}...')
kproto_final = KPrototypes(n_clusters=k_optimal, init='Cao', random_state=42, n_jobs=-1)
clusters = kproto_final.fit_predict(X_scaled, categorical=cat_idx)

# Attach cluster labels back to the ORIGINAL (unscaled) dataframe for interpretability
df_result = df[num_cols + cat_cols].copy()
df_result['cluster'] = clusters
df_result['y'] = target

print(f'\nCluster size distribution:')
print(df_result['cluster'].value_counts().sort_index())

## Step 7 — Conversion Rate per Cluster ⭐ (Key Insight)

This is the most important step for business interpretation.
We cross-tabulate cluster labels with the target `y` to find which clusters
have the highest subscription rate.

**How to read this:**
- The dataset baseline conversion rate is ~11%
- Any cluster with rate >> 11% is a **high-value segment** to prioritize
- Any cluster with rate << 11% should receive lower campaign priority

The `normalize='index'` argument in crosstab gives row-wise proportions
(i.e., conversion rate within each cluster).

In [ ]:
# Conversion rate per cluster
conversion = df_result.groupby('cluster')['y'].agg(
    conversion_rate='mean',
    cluster_size='count'
).reset_index()
conversion['conversion_rate_pct'] = (conversion['conversion_rate'] * 100).round(1)
conversion['share_of_total'] = (conversion['cluster_size'] / len(df_result) * 100).round(1)
conversion = conversion.sort_values('conversion_rate_pct', ascending=False)

print('=== Conversion Rate by Cluster ===')
print(conversion[['cluster','conversion_rate_pct','cluster_size','share_of_total']].to_string(index=False))
print(f'\nBaseline (overall): {target.mean()*100:.1f}%')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(conversion['cluster'].astype(str), conversion['conversion_rate_pct'],
            color=['#2ecc71' if r > target.mean()*100 else '#e74c3c'
                   for r in conversion['conversion_rate_pct']])
axes[0].axhline(y=target.mean()*100, color='navy', linestyle='--', label='Baseline')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_title('Subscription Rate per Cluster')
axes[0].legend()

axes[1].pie(conversion['cluster_size'], labels=[f'Cluster {c}' for c in conversion['cluster']],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Cluster Size Distribution')

plt.tight_layout()
plt.savefig('conversion_by_cluster.png', dpi=150)
plt.show()

## Step 8 — Profile Each Cluster

To understand *who* is in each cluster, we compute:
- **Numeric profile**: mean value per cluster (e.g., average age, average call duration)
- **Categorical profile**: mode (most common value) per cluster (e.g., most common job type)

Reading the profile alongside conversion rate lets you name each segment meaningfully.

In [ ]:
# Numeric profile — use original unscaled values for readability
num_profile = df_result.groupby('cluster')[num_cols].mean().round(2)

# Categorical profile — use mode (most frequent value)
cat_profile = df_result.groupby('cluster')[cat_cols].agg(
    lambda x: x.value_counts().index[0]
)

full_profile = pd.concat([num_profile, cat_profile], axis=1)

print('=== Full Cluster Profile (numeric=mean, categorical=mode) ===')
print(full_profile.T.to_string())

## Step 9 — Heatmap of Numeric Profile

The heatmap shows normalized numeric feature values per cluster (color scale),
with actual mean values annotated inside each cell.

**How to read it:**
- Darker red = higher value relative to other clusters for that feature
- Lighter yellow = lower value
- Look for clusters with high `duration` + high `previous` contacts — these tend to be high converters

In [ ]:
# Normalize numeric profile (0–1) for color scale, keep raw values for annotation
num_profile_norm = (num_profile - num_profile.min()) / (num_profile.max() - num_profile.min())

plt.figure(figsize=(14, 6))
sns.heatmap(
    num_profile_norm.T,
    annot=num_profile.T,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Normalized Value'}
)
plt.title('Cluster Numeric Profile\n(color = normalized, annotation = actual mean)', fontsize=13)
plt.xlabel('Cluster', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=150)
plt.show()

## Step 10 — Name Your Segments

After reviewing the profiles and conversion rates above, assign business-friendly names
to each cluster. This is the most critical step for storytelling in a portfolio or interview.

**Naming tips:**
- Use age + job as the primary descriptor
- Include the conversion behavior (High/Low/Moderate)
- Example: `'Retired High-Balance Subscriber'`, `'Young Blue-Collar Low Converter'`

> Update the `segment_names` dictionary based on what YOU observe in your profile above.

In [ ]:
# ✏️ UPDATE THESE based on your profile output above
segment_names = {
    0: 'Cluster 0 — [Your Label Here]',
    1: 'Cluster 1 — [Your Label Here]',
    2: 'Cluster 2 — [Your Label Here]',
    3: 'Cluster 3 — [Your Label Here]',
}

df_result['segment'] = df_result['cluster'].map(segment_names)

# Final summary
summary = df_result.groupby('segment').agg(
    size=('y', 'count'),
    conversion_rate=('y', lambda x: f"{x.mean()*100:.1f}%")
).reset_index()
print('=== Final Segment Summary ===')
print(summary.to_string(index=False))

# Export
df_result.to_csv('clustered_customers.csv', index=False)
print('\nResults saved to clustered_customers.csv')

## Summary & Business Recommendation

Use this section to write your final narrative after running the notebook.

**Template:**

> *"K-Prototypes identified [k] distinct customer segments in the UCI Bank Marketing dataset.
> Cluster [X] — characterized by [age group], [job type], and [previous contact outcome] —
> shows a [XX]% subscription rate, compared to the [11]% baseline.
> The bank should concentrate its calling resources on this segment to maximize campaign ROI.
> Cluster [Y], with only [Z]% conversion, represents [XX]% of total contacts and should
> be deprioritized or approached with a different strategy."*

---

### Key Files Generated
| File | Description |
|---|---|
| `elbow_kproto.png` | Elbow curve to justify k selection |
| `conversion_by_cluster.png` | Conversion rate bar chart + cluster size pie |
| `cluster_heatmap.png` | Numeric feature heatmap per cluster |
| `clustered_customers.csv` | Full dataset with cluster and segment labels |
